In [19]:

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [20]:

df = pd.read_csv('Salary_Data.csv')
df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,Male,Bachelor's,Software Engineer,5.0,90000.0
1,28.0,Female,Master's,Data Analyst,3.0,65000.0
2,45.0,Male,PhD,Senior Manager,15.0,150000.0
3,36.0,Female,Bachelor's,Sales Associate,7.0,60000.0
4,52.0,Male,Master's,Director,20.0,200000.0


In [21]:
df.isnull().sum()

Age                    2
Gender                 2
Education Level        2
Job Title              2
Years of Experience    2
Salary                 2
dtype: int64

In [22]:
## drop rows where have null values
df.dropna(inplace=True)
df.isnull().sum()

Age                    0
Gender                 0
Education Level        0
Job Title              0
Years of Experience    0
Salary                 0
dtype: int64

In [23]:
cols_to_encode = ['Gender', 'Education Level', 'Job Title']
encoders = {}

for col in cols_to_encode:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])

df.head()

,Age,Gender,Education Level,Job Title,Years of Experience,Salary
0,32.0,1,0,159,5.0,90000.0
1,28.0,0,1,17,3.0,65000.0
2,45.0,1,2,130,15.0,150000.0
3,36.0,0,0,101,7.0,60000.0
4,52.0,1,1,22,20.0,200000.0


In [24]:
X = df.drop(columns=['Salary'])
y = df['Salary'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)

(298, 5) (298,)


In [25]:
scaler = StandardScaler()

cols_to_scale = ['Age', 'Years of Experience']

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale]  = scaler.transform(X_test[cols_to_scale])

X_train.head()

,Age,Gender,Education Level,Job Title,Years of Experience
193,-0.496349,1,0,141,-0.469123
75,-0.069680,1,0,96,-0.007995
84,-1.207465,0,0,56,-1.237669
363,-0.638572,1,0,65,-0.776541
16,-0.638572,0,1,83,-0.469123


In [26]:

X_train_tensor = torch.from_numpy(X_train.values).float()
X_test_tensor  = torch.from_numpy(X_test.values).float()

# ✅ float32 not long (regression)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.float32)

In [27]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):

    def __init__(self, features, labels):
        self.features = features
        self.labels   = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]


In [28]:
# Create dataset objects
train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset  = CustomDataset(X_test_tensor,  y_test_tensor)

train_dataset[0]

(tensor([ -0.4963,   1.0000,   0.0000, 141.0000,  -0.4691]), tensor(95000.))

In [29]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

In [30]:
# Model
class MySimpleNN(nn.Module):

    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)    # ✅ 1 output, NO activation (regression)
        )

    def forward(self, features):
        return self.network(features)

In [31]:
epochs        = 100
learning_rate = 0.01

In [32]:
model     = MySimpleNN(num_features=X_train_tensor.shape[1])

loss_fn   = nn.MSELoss()    # ✅ MSE for regression

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [33]:
#tranning loop
for epoch in range(epochs):

    total_epoch_loss = 0

    for batch_features, batch_labels in train_loader:

        y_pred = model(batch_features)

        loss = loss_fn(y_pred.squeeze(), batch_labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_epoch_loss += loss.item()

    avg_loss = total_epoch_loss / len(train_loader)
    print(f'Epoch: {epoch + 1}, Loss: {avg_loss:.2f}')

Epoch: 1, Loss: 12403097190.40
Epoch: 2, Loss: 11503544217.60
Epoch: 3, Loss: 7726549632.00
Epoch: 4, Loss: 4098516224.00
Epoch: 5, Loss: 4034806336.00
Epoch: 6, Loss: 3521074470.40
Epoch: 7, Loss: 3470066124.80
Epoch: 8, Loss: 3567200460.80
Epoch: 9, Loss: 3221835308.80
Epoch: 10, Loss: 3263924736.00
Epoch: 11, Loss: 3271662617.60
Epoch: 12, Loss: 3134345740.80
Epoch: 13, Loss: 2730474188.80
Epoch: 14, Loss: 2706170432.00
Epoch: 15, Loss: 2389797235.20
Epoch: 16, Loss: 2006297324.80
Epoch: 17, Loss: 1575287270.40
Epoch: 18, Loss: 1018400595.20
Epoch: 19, Loss: 626520422.40
Epoch: 20, Loss: 438289413.60
Epoch: 21, Loss: 367853633.60
Epoch: 22, Loss: 338925995.20
Epoch: 23, Loss: 322763403.20
Epoch: 24, Loss: 306310915.20
Epoch: 25, Loss: 318133652.80
Epoch: 26, Loss: 326690886.40
Epoch: 27, Loss: 286487555.20
Epoch: 28, Loss: 315921040.00
Epoch: 29, Loss: 275366920.00
Epoch: 30, Loss: 307310048.00
Epoch: 31, Loss: 296614176.80
Epoch: 32, Loss: 282613537.60
Epoch: 33, Loss: 273157798.40

In [40]:
# Evaluation using MAE
model.eval()
predictions_list = []
actuals_list     = []

with torch.inference_mode():

    for batch_features, batch_labels in test_loader:

        y_pred = model(batch_features)
        predictions_list.extend(y_pred.squeeze().tolist())
        actuals_list.extend(batch_labels.tolist())

predictions = np.array(predictions_list)
actuals     = np.array(actuals_list)

mae = np.mean(np.abs(predictions - actuals))
print(f'MAE : ${mae:,.2f}')
print(f'Average Salary: ${actuals.mean():,.2f}')
print(f'Actual list: {actuals_list[:5]}')
print(f'Predicted list: {predictions_list[:5]}')

MAE : $10,504.06
Average Salary: $102,466.67
Actual list: [180000.0, 65000.0, 125000.0, 80000.0, 140000.0]
Predicted list: [181237.65625, 81858.9609375, 112663.734375, 87163.46875, 151151.140625]


In [41]:
for i, title in enumerate(encoder.classes_):
    print(f'{i} → {title}')

0 → Account Manager
1 → Accountant
2 → Administrative Assistant
3 → Business Analyst
4 → Business Development Manager
5 → Business Intelligence Analyst
6 → CEO
7 → Chief Data Officer
8 → Chief Technology Officer
9 → Content Marketing Manager
10 → Copywriter
11 → Creative Director
12 → Customer Service Manager
13 → Customer Service Rep
14 → Customer Service Representative
15 → Customer Success Manager
16 → Customer Success Rep
17 → Data Analyst
18 → Data Entry Clerk
19 → Data Scientist
20 → Digital Content Producer
21 → Digital Marketing Manager
22 → Director
23 → Director of Business Development
24 → Director of Engineering
25 → Director of Finance
26 → Director of HR
27 → Director of Human Capital
28 → Director of Human Resources
29 → Director of Marketing
30 → Director of Operations
31 → Director of Product Management
32 → Director of Sales
33 → Director of Sales and Marketing
34 → Event Coordinator
35 → Financial Advisor
36 → Financial Analyst
37 → Financial Manager
38 → Graphic Des

In [42]:
# ───────────────────────────────────────────────
# CELL 18 — Predict for new employee
# ───────────────────────────────────────────────

# Column order: Age, Gender, Education Level, Job Title, Years of Experience

# Gender          → Female=0,     Male=1
# Education Level → check encoder (alphabetical)
# Job Title       → check Cell 17 output above

new_employee = pd.DataFrame([{
    'Age': 30,
    'Gender': 1,
    'Education Level': 2,
    'Job Title': 50,
    'Years of Experience': 5,
}])

# scale Age and Years of Experience using the same feature names as during fit
new_employee[['Age', 'Years of Experience']] = scaler.transform(
    new_employee[['Age', 'Years of Experience']]
)

input_tensor = torch.from_numpy(new_employee.to_numpy(dtype=np.float32))


c:\Users\gamin\OneDrive\Desktop\DL_Projects\env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [43]:
#prediction
model.eval()
with torch.inference_mode():
    output           = model(input_tensor)
    predicted_salary = output.item()    # ✅ directly float, no argmax, no threshold

print(f'Predicted Salary : ${predicted_salary:,.2f}')

Predicted Salary : $85,051.43
